In [3]:
import pandas as pd
import numpy as np
from decimal import Decimal, getcontext

# Set precision for Decimal operations
getcontext().prec = 22

def format_currency(val):
    try:
        if pd.isna(val) or val == '': return Decimal('0.000000')
        return Decimal(str(val)).quantize(Decimal('1.000000'))
    except:
        return Decimal('0.000000')

# Define file paths
rent_path = r'C:\Users\user\Desktop\BusinessCaseStudy\RentRoll_sample.xlsx'
disc_path = r'C:\Users\user\Desktop\BusinessCaseStudy\Discounts_sample.xlsx'
move_path = r'C:\Users\user\Desktop\BusinessCaseStudy\MoveInsAndMoveOuts_sample.xlsx'

def process_redbox_data():
    print("Loading Excel files...")
    # Using openpyxl to read Excel files
    df_rent = pd.read_excel(rent_path)
    df_disc = pd.read_excel(disc_path)
    df_move = pd.read_excel(move_path)

    # 1. Parsing sTypeName
    cat_columns = ['Size Category', 'Unit Type', 'Size Range', 'Shape', 
                   'Door Type', 'Pillar', 'Pricing Tier']
    if 'sTypeName' in df_rent.columns:
        df_rent[cat_columns] = df_rent['sTypeName'].str.split('/', expand=True)

    # 2. Revenue Override Logic
    # NOTE: Joining on 'sUnit' as a reliable common key across files
    df_merged = pd.merge(df_rent, df_disc[['sUnit', 'dcAmt']], on='sUnit', how='left')

    # Apply Business Logic: Vacant ($0) -> Discount (dcAmt) -> Standard (dcRent)
    def calculate_rent(row):
        if pd.isna(row.get('TenantID')): return format_currency(0)
        elif pd.notna(row['dcAmt']): return format_currency(row['dcAmt'])
        else: return format_currency(row['dcRent'])

    df_merged['Actual_Monthly_Rent'] = df_merged.apply(calculate_rent, axis=1)

    # 3. Standardize Timestamps
    date_cols = ['dCreated', 'dDeleted', 'dUpdated', 'dLeaseDate']
    for col in date_cols:
        if col in df_merged.columns:
            df_merged[col] = pd.to_datetime(df_merged[col], errors='coerce').dt.strftime('%Y-%m-%d')

    print("Pipeline processing successful.")
    return df_merged, df_move

# Execute and Save
try:
    cleaned_rent, move_data = process_redbox_data()
    cleaned_rent.to_excel(r'C:\Users\user\Desktop\BusinessCaseStudy\Cleaned_RentRoll.xlsx', index=False)
    print("Successfully saved Cleaned_RentRoll.xlsx")
except Exception as e:
    print(f"An error occurred: {e}")

Loading Excel files...
Pipeline processing successful.
Successfully saved Cleaned_RentRoll.xlsx


In [4]:
import pandas as pd
import numpy as np

def run_phase_2_operational_analysis(rent_data, move_data):
    # Ensure dates are datetime objects
    move_data['MoveDate'] = pd.to_datetime(move_data['MoveDate'])
    
    # 1. Net Absorption
    # Definition: Change in occupied area over the period Nov 1 - Nov 12, 2025
    start_date, end_date = '2025-11-01', '2025-11-12'
    period_data = move_data[(move_data['MoveDate'] >= start_date) & (move_data['MoveDate'] <= end_date)]
    net_abs = period_data['MovedInArea'].sum() - period_data['MovedOutArea'].sum()
    
    # 2. Yield Efficiency
    # Definition: Actual Revenue per square foot per Unit Type
    # We group by Unit Type to calculate average rent density
     #yield_df = rent_data.groupby('Unit Type').agg({
     #    'Actual_Monthly_Rent': 'mean',
     #    'sSize': 'first' # Using this to map back to area if needed
     #})
    
    # 2. Yield Efficiency
    # Definition: Actual Revenue per square foot per Unit Type
    # 2.1. Calculate 'Area' from the 'sSize' column
    # We split by 'x', convert to floats, and multiply
    rent_data[['Width', 'Height']] = rent_data['sSize'].str.split('x', expand=True).astype(float)
    rent_data['Area'] = rent_data['Width'] * rent_data['Height']
    
    # 2.2. NOW run the Yield Efficiency aggregation
    # This will no longer trigger the KeyError because 'Area' now exists
    # We group by Unit Type to calculate average rent AND average area
    yield_df = rent_data.groupby('Unit Type').agg({
        'Actual_Monthly_Rent': 'mean',
        'Area': 'mean'  # Make sure this matches your actual column name (e.g., 'Area', 'Sqft', or 'MovedInArea')
    })

    # 2.3. Calculate the Yield
    yield_df['Yield_Per_Sqft'] = yield_df['Actual_Monthly_Rent'] / yield_df['Area']

    print("Yield Efficiency calculated successfully.")
    #print(yield_df.sort_values(by='Yield_Per_Sqft', ascending=False))
    
    # 3. Promotion Impact
    # Definition: Stickiness of tenants by discount plan (Avg Length of Stay)
    move_outs = move_data[move_data['MoveOut'] == 1]
    # Fill empty discount plans with a placeholder string before grouping
    move_data['sDiscountPlan'] = move_data['sDiscountPlan'].fillna('Standard / No Discount')
    promo_impact = move_outs.groupby('sDiscountPlan')['MovedOutDaysRented'].mean()
  
    # --- Output to Console ---
    print("=== PART 1: NET ABSORPTION ===")
    print(f"Net square footage change: {net_abs:.2f} sqft\n")
    
    print("=== PART 2: YIELD EFFICIENCY ===")
    print(yield_df, "\n")
    print(yield_df.sort_values(by='Yield_Per_Sqft', ascending=False))
    
    print("=== PART 3: PROMOTION IMPACT ===")
    print(promo_impact.sort_values(ascending=False))
    
    return net_abs, yield_df, promo_impact

# Execute analysis
net_abs, yield_metrics, promo_metrics = run_phase_2_operational_analysis(cleaned_rent, move_data)

Yield Efficiency calculated successfully.
=== PART 1: NET ABSORPTION ===
Net square footage change: 43.30 sqft

=== PART 2: YIELD EFFICIENCY ===
          Actual_Monthly_Rent       Area Yield_Per_Sqft
Unit Type                                              
FB1               1195.295838  51.299306      23.300429
FB2                    1396.0  63.722222      21.907585
LU                 144.992105   9.764211      14.849342
RG                1211.039904  36.881649       32.83584 

          Actual_Monthly_Rent       Area Yield_Per_Sqft
Unit Type                                              
RG                1211.039904  36.881649       32.83584
FB1               1195.295838  51.299306      23.300429
FB2                    1396.0  63.722222      21.907585
LU                 144.992105   9.764211      14.849342
=== PART 3: PROMOTION IMPACT ===
sDiscountPlan
10% Discount                1266.0
2023 Red Hot -40% online     875.0
2024 Summer Flash -40%       402.0
Prepay 50% off               